In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText


In [ ]:
# ----------------------------
# Load
# ----------------------------
df = pd.read_csv("../results/NEON-CPU.csv")

# Full strategy name
df["strategy_full"] = (
    df["strategy"]
    + " ("
    + df["julia_or_neon"]
    + ")"
)

# ----------------------------
# Convert time -> milliseconds
# ----------------------------
df["ms_mean"] = df["time_us"] / 1e3

# Effective bandwidth [GB/s]
# nnz * 24 bytes / runtime
df["bandwidth"] = df["nnz"] * 24  / (df["ms_mean"] * 1000000) 

# Equivalent:
# df["bandwidth"] = df["nnz"] * 24 / (df["time_us"] / 1e6)

# ----------------------------
# Aggregate repetitions
# ----------------------------
agg = (
    df.groupby(
        [
            "strategy_full",
            "threads",
            "node",
            "executor",
        ],
        as_index=False,
    )
    .agg(
        bandwidth=("bandwidth", "median")
    )
)

# ----------------------------
# Normalize against threads=1
# per strategy / node
# ----------------------------
baseline = (
    agg[agg["threads"] == 1]
    [["strategy_full", "node", "executor", "bandwidth"]]
    .rename(
        columns={
            "bandwidth": "bw_1"
        }
    )
)

# ----------------------------
# Normalize against the serial implementation
# per node
# ----------------------------
serial_baseline_strategy = "Face-Based (JuNe)"
serial_baseline = (
    agg[
        (agg["threads"] == 1)
        & (agg["strategy_full"] == serial_baseline_strategy)
    ]
    [["node", "executor", "bandwidth"]]
    .rename(
        columns={
            "bandwidth": "bw_serial"
        }
    )
)

plot_df = (
    agg.merge(
        baseline,
        on=[
            "strategy_full",
            "node",
            "executor",
        ],
    )
    .merge(
        serial_baseline,
        on=[
            "node",
            "executor",
        ],
    )
)

plot_df["speedup"] = (
    plot_df["bandwidth"]
    / plot_df["bw_1"]
)
plot_df["speedup_vs_serial_bandwidth"] = (
    plot_df["bandwidth"]
    / plot_df["bw_serial"]
)

plot_df = plot_df[
    [
        "strategy_full",
        "threads",
        "node",
        "executor",
        "bandwidth",
        "bw_1",
        "bw_serial",
        "speedup",
        "speedup_vs_serial_bandwidth",
    ]
]


In [ ]:

# ----------------------------
# Plot
# ----------------------------
sns.set_theme(rc={'figure.figsize':(20, 12)})
sns.set_style("whitegrid")
sns.color_palette("deep", 8)

nodes = sorted(plot_df["node"].unique())
threads = sorted(plot_df["threads"].unique())
thread_positions = {thread: pos for pos, thread in enumerate(threads)}
speedup_plot_df = plot_df.assign(
    thread_pos=plot_df["threads"].map(thread_positions)
)

# for ax, node in zip(axes, nodes):
with sns.plotting_context("paper", font_scale=1.7):
    p = sns.relplot(
        # data=df[(df["threads"]==128) ],
        data=speedup_plot_df,
        x="thread_pos",
        y="speedup",
        kind="line",
        col="node",
        col_order=["gpu-nvidia-h100", "gpu-nvidia-h200"],
        hue="strategy_full",
        style="strategy_full",
        markersize=12,
        markers=True
    )

    # Ideal scaling
    thread_x = list(range(len(threads)))
    for ax in p.axes.flat:
        ax.plot(
            thread_x,
            threads,
            "--",
            color="black",
            alpha=0.5,
            linewidth=1.5,
            label="Ideal",
        )

    p.set(
        yscale="log",
        xlabel="#Threads",
        ylabel="Speedup w.r.t Assembly Bandwidth",
    )
    for ax in p.axes.flat:
        ax.set_xticks(thread_x)
        ax.set_xticklabels(threads)
        ax.set_xlim(thread_x[0] - 0.3, thread_x[-1] + 0.3)

    desired_order = [
        "Cell-Based (NeoN)",
        "Cell-Based (JuNe)",
        "Global Face-Based (NeoN)",
        "Global Face-Based (JuNe)",
        "Face-Based (NeoN)",
        "Face-Based (JuNe)",
        "Fused Face-Based (NeoN)",
        "Fused Cell-Based (NeoN)"
    ]

    handles, labels = p.axes.flat[0].get_legend_handles_labels()
    if p._legend is not None:
        p._legend.remove()
        p._legend = None
    print(f"labels: {labels}")
    by_label = dict(zip(labels, handles))

    ordered_labels = [label for label in desired_order if label in by_label]
    ordered_handles = [by_label[label] for label in ordered_labels]
    
    at = AnchoredText(
        "INTEL XEON",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[0].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[0].add_artist(at)  
    at = AnchoredText(
        "AMD EPYC",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[1].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[1].add_artist(at)    
    p.set_titles("")

    p.fig.legend(
        ordered_handles,
        ordered_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=4,
        # title="Fused Strategy",
        frameon=False,
    )
    p.fig.subplots_adjust(
        left=0.07,
        right=0.98,
        # bottom=0.18,
        top=0.78,
        # wspace=0.1,
    )
plt.savefig("cpu-speedup-bandwidth.svg")

In [ ]:
###### EDIT THIS
# ----------------------------
# Plot
# ----------------------------
sns.set_theme(rc={'figure.figsize':(20, 12)})
sns.set_style("whitegrid")
sns.color_palette("deep", 8)

bandwidth_source_df = pd.read_csv("../results/NEON-CPU.csv")
bandwidth_source_df["strategy_full"] = (
    bandwidth_source_df["strategy"]
    + " ("
    + bandwidth_source_df["julia_or_neon"]
    + ")"
)
bandwidth_source_df["ms_mean"] = bandwidth_source_df["time_us"] / 1e3
bandwidth_source_df["bandwidth"] = bandwidth_source_df["nnz"] * 24 / (bandwidth_source_df["ms_mean"] * 1000000)
bandwidth_plot_df = bandwidth_source_df[bandwidth_source_df["threads"] == 1].copy()
nodes = sorted(bandwidth_plot_df["node"].unique())

# for ax, node in zip(axes, nodes):
with sns.plotting_context("paper", font_scale=1.7):
    p = sns.relplot(
        data=bandwidth_plot_df,
        x="cells",
        y="bandwidth",
        kind="line",
        col="node",
        col_order=["gpu-nvidia-h100", "gpu-nvidia-h200"],
        hue="strategy_full",
        style="strategy_full",
        err_style=None,
        markersize=10,
        markers=True
    )

    # Ideal scaling
    threads = sorted(
        bandwidth_source_df["threads"].unique()
    )

    p.set(
        yscale="log",
        xscale="log",
        xlabel="#Threads",
        ylabel="Matrix Assembly Bandwidth (GB/s)",
    )
    ytick_strategies = [
        "Fused Cell-Based (NeoN)",
        "Global Face-Based (NeoN)",
    ]
    line_bandwidth = bandwidth_plot_df.groupby(
        ["node", "strategy_full", "cells"],
        as_index=False,
    )["bandwidth"].mean()
    peak_yticks = line_bandwidth[
        line_bandwidth["strategy_full"].isin(ytick_strategies)
    ].groupby(["node", "strategy_full"])["bandwidth"].max().to_numpy()
    min_yticks = line_bandwidth.groupby("node")["bandwidth"].min().to_numpy()
    extra_yticks = np.concatenate([peak_yticks, min_yticks])
    for ax in p.axes.flat:
        ymin, ymax = ax.get_ylim()
        custom_yticks = [round(float(tick), 3) for tick in extra_yticks if ymin <= tick <= ymax]
        ytick_values = sorted(set(custom_yticks))
        ax.set_yticks(ytick_values)
        ax.set_yticklabels([
            f"{tick:.2f}"
            for tick in ytick_values
        ])
    p.axes.flat[0].tick_params(axis="y", which="both", labelleft=True)
    p.axes.flat[1].tick_params(axis="y", which="both", labelleft=False)

    desired_order = [
        "Cell-Based (NeoN)",
        "Cell-Based (JuNe)",
        "Global Face-Based (NeoN)",
        "Global Face-Based (JuNe)",
        "Face-Based (NeoN)",
        "Face-Based (JuNe)",
        "Fused Face-Based (NeoN)",
        "Fused Cell-Based (NeoN)"
    ]
    handles, labels = p.axes.flat[0].get_legend_handles_labels()
    if p._legend is not None:
        p._legend.remove()
        p._legend = None
    print(f"labels: {labels}")
    by_label = dict(zip(labels, handles))

    ordered_labels = [label for label in desired_order if label in by_label]
    ordered_handles = [by_label[label] for label in ordered_labels]
    
    at = AnchoredText(
        "INTEL XEON",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[0].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[0].add_artist(at)  
    at = AnchoredText(
        "AMD EPYC",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[1].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[1].add_artist(at)    
    p.set_titles("")

    p.fig.legend(
        ordered_handles,
        ordered_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=4,
        # title="Fused Strategy",
        frameon=False,
    )
    p.fig.subplots_adjust(
        left=0.07,
        right=0.98,
        # bottom=0.18,
        top=0.78,
        # wspace=0.1,
    )

plt.savefig("cpu-bandwidth-1.svg")


In [ ]:
plot_df.to_csv("plotdt.csv")


In [ ]:
# ----------------------------
# Load
# ----------------------------
df = pd.read_csv("../results/NEON-GPU.csv")

# Full strategy name
df["strategy_full"] = (
    df["strategy"]
    + " ("
    + df["julia_or_neon"]
    + ")"
)

# ----------------------------
# Convert time -> milliseconds
# ----------------------------
df["ms_mean"] = df["time_us"] / 1e3

# Effective bandwidth [GB/s]
# nnz * 24 bytes / runtime
df["bandwidth"] = df["nnz"] * 24  / (df["ms_mean"] * 1000000) 

# Equivalent:
# df["bandwidth"] = df["nnz"] * 24 / (df["time_us"] / 1e6)

# ----------------------------
# Aggregate repetitions
# ----------------------------
agg = (
    df.groupby(
        [
            "strategy_full",
            "threads",
            "node",
            "executor",
        ],
        as_index=False,
    )
    .agg(
        bandwidth=("bandwidth", "median")
    )
)



In [ ]:

# ----------------------------
# Plot
# ----------------------------
sns.set_theme(rc={'figure.figsize':(20, 12)})
sns.set_style("whitegrid")
sns.color_palette("deep", 8)

nodes = sorted(plot_df["node"].unique())

# for ax, node in zip(axes, nodes):
with sns.plotting_context("paper", font_scale=1.7):
    p = sns.relplot(
        data=df[(df["threads"]==1) ],
        x="cells",
        y="bandwidth",
        kind="line",
        col="node",
        col_order=["gpu-nvidia-h100", "gpu-nvidia-h200"],
        hue="strategy_full",
        style="strategy_full",
        err_style=None,
        markersize=10,
        markers=True
    )

    # Ideal scaling
    threads = sorted(
        subset["threads"].unique()
    )

    p.set(
        # yscale="log",
        xscale="log",
        xlabel="#Threads",
        ylabel="Matrix Assembly Bandwidth (GB/s)",
    )

    desired_order = [
        "Cell-Based (NeoN)",
        "Cell-Based (JuNe)",
        "Global Face-Based (NeoN)",
        "Global Face-Based (JuNe)",
        "Face-Based (NeoN)",
        "Face-Based (JuNe)",
        "Fused Face-Based (NeoN)",
        "Fused Cell-Based (NeoN)"
    ]
    handles, labels = p.axes.flat[0].get_legend_handles_labels()
    if p._legend is not None:
        p._legend.remove()
        p._legend = None
    print(f"labels: {labels}")
    by_label = dict(zip(labels, handles))

    ordered_labels = [label for label in desired_order if label in by_label]
    ordered_handles = [by_label[label] for label in ordered_labels]
    
    at = AnchoredText(
        "INTEL XEON",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[0].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[0].add_artist(at)  
    at = AnchoredText(
        "AMD EPYC",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[1].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[1].add_artist(at)    
    p.set_titles("")

    p.fig.legend(
        ordered_handles,
        ordered_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=4,
        # title="Fused Strategy",
        frameon=False,
    )
    p.fig.subplots_adjust(
        left=0.07,
        right=0.98,
        # bottom=0.18,
        top=0.78,
        # wspace=0.1,
    )

plt.savefig("cpu-bandwidth-1.svg")
